# TP 7 : La Fuite de Données (Data Leakage)
(SUJET GUIDÉ)

**Dataset :** Diabetes 130-US Hospitals (UCI ML Repository)  
**Contexte :** 101 766 hospitalisations dans 130 hôpitaux américains (1999-2008).  
**Tâche :** Prédire si un patient diabétique sera **réadmis dans les 30 jours** suivant sa sortie.

---

## Qu'est-ce que la fuite de données ?

La **fuite de données** (*data leakage*) se produit quand des informations du **jeu de test** (ou du futur) contaminent l'entraînement. Le modèle semble excellent... mais il **triche** !

### Les 3 types étudiés ici :
| # | Type | Exemple concret dans ce TP |
|---|---|---|
| 1 | **Fuite de prétraitement** | Normalisation calculée sur tout le dataset avant le split |
| 2 | **Fuite d'encodage cible** | Statistiques par groupe calculées sur le dataset complet |
| 3 | **Fuite patient** | Le même patient apparaît dans train ET test (102k séjours, 72k patients) |

In [ ]:
# Installer le package UCI ML Repository si nécessaire
# !pip install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, cross_val_score, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 10)

---
## Chargement et Préparation des Données

In [ ]:
# Chargement depuis UCI ML Repository
print("Chargement du dataset diabetes-130...")
dataset = fetch_ucirepo(id=296)
df_raw = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

print(f"Dimensions : {df_raw.shape}")
print(f"\nColonnes : {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# Préparation minimale commune à tous les scénarios
df = df_raw.copy()

# Cible binaire : réadmission dans les 30 jours (1) ou non (0)
df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

# Remplacer '?' par NaN (valeurs manquantes encodées comme texte dans ce dataset)
df.replace('?', np.nan, inplace=True)

# Features numériques disponibles sans transformation complexe
FEATS_NUM = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

# Sous-dataset propre (features numériques uniquement, sans NaN)
df_num = df[FEATS_NUM + ['readmit_30', 'patient_nbr']].dropna().copy()

print(f"Dataset de travail : {len(df_num)} lignes")
print(f"Taux de réadmission <30j : {df_num['readmit_30'].mean():.2%}")
print(f"Patients uniques : {df_num['patient_nbr'].nunique()}")
df_num[FEATS_NUM].describe().round(2)

---
## Scénario 1 : Fuite de Prétraitement

**Le piège classique.**

La normalisation (StandardScaler) calcule la **moyenne** et l'**écart-type** de chaque feature.  
Si on la calcule sur l'ensemble du dataset *avant* de séparer train/test, les statistiques du test influencent le prétraitement → **fuite**.

Dans un contexte hospitalier réel, les données d'un nouveau patient arrivant en urgence ne font pas encore partie du dataset → impossible de connaître la nouvelle moyenne.

### Analogy
> Le laboratoire d'analyses normalise les résultats en utilisant les paramètres calculés sur des patients *y compris ceux dont il doit encore établir le diagnostic*.

In [ ]:
X = df_num[FEATS_NUM].values
y = df_num['readmit_30'].values

# ❌ MAUVAIS : normaliser avant de splitter
scaler_bad = StandardScaler()
X_scaled_all = scaler_bad.fit_transform(X)   # ← Le scaler "voit" les données de test !

X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_scaled_all, y, test_size=0.2, random_state=42, stratify=y
)

model_bad = LogisticRegression(max_iter=500, class_weight='balanced')
model_bad.fit(X_train_bad, y_train)
f1_bad = f1_score(y_test, model_bad.predict(X_test_bad))
print(f"F1 (AVEC fuite de prétraitement) : {f1_bad:.4f}")

In [ ]:
# TODO : Corrigez la fuite

# Règle : split D'ABORD, puis fit_transform sur le train, transform sur le test

# 1. Séparez X et y (test_size=0.2, random_state=42, stratify=y)
# X_train_raw, X_test_raw, y_train, y_test = ...

# 2. Créez un scaler et ajustez-le UNIQUEMENT sur le train
# scaler_good = StandardScaler()
# X_train_good = ...
# X_test_good  = ...   # transform SANS fit !

# 3. Entraînez le même modèle et calculez le F1
# model_good = LogisticRegression(max_iter=500, class_weight='balanced')
# ...
# f1_good = ...
# print(f"F1 (SANS fuite) : {f1_good:.4f}")

In [ ]:
# ✅ Solution propre : utiliser un Pipeline
# Un Pipeline garantit que le scaler est refitté correctement à chaque fold de cross-validation.

# TODO : Créez un Pipeline [('scaler', StandardScaler()), ('model', LogisticRegression(...))]
#        Évaluez avec cross_val_score (cv=5, scoring='f1')
#        Comparez les 3 F1 dans un graphique à barres

# pipe = Pipeline([...])
# scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
# f1_pipe = scores.mean()
# print(f"F1 (Pipeline cross-val) : {f1_pipe:.4f}")

---
## Scénario 2 : Fuite d'Encodage Cible (Target Encoding Leakage)

La variable `age` est encodée par tranches de 10 ans (`[0-10)`, `[10-20)`, ..., `[90-100)`).  
Pour l'utiliser dans un modèle, une technique populaire est le **target encoding** : remplacer chaque modalité par le **taux moyen de réadmission** de ce groupe.

**Le piège :** si on calcule ce taux sur l'ensemble du dataset avant le split, les patients du jeu de test ont influencé l'encodage → **fuite**.

### Pourquoi c'est grave ?
Les statistiques du test contaminent le train. L'encodage "connaît" la réponse des patients de test.

In [ ]:
# Explorons la variable age et son lien avec la réadmission
df_age = df[['age', 'readmit_30']].dropna().copy()

taux_par_age = df_age.groupby('age')['readmit_30'].agg(['mean', 'count'])
taux_par_age.columns = ['taux_readmission', 'n_patients']
taux_par_age = taux_par_age.sort_index()

print("Taux de réadmission par tranche d'âge :")
print(taux_par_age.round(4))

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(taux_par_age.index, taux_par_age['taux_readmission'] * 100,
       color='steelblue', alpha=0.8)
ax.set_xlabel("Tranche d'âge")
ax.set_ylabel("Taux de réadmission <30j (%)")
ax.set_title("Taux de réadmission par tranche d'âge (dataset complet)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Préparons un dataset avec les features numériques + age encodé
df_s2 = df[FEATS_NUM + ['age', 'readmit_30']].dropna().copy()

# ❌ MAUVAIS : Target encoding calculé sur TOUT le dataset
encoding_global = df_s2.groupby('age')['readmit_30'].mean()  # ← Voit le test !
df_s2['age_encoded_leak'] = df_s2['age'].map(encoding_global)

feats_avec_fuite = FEATS_NUM + ['age_encoded_leak']
X_leak = df_s2[feats_avec_fuite].values
y_s2   = df_s2['readmit_30'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, y_s2, test_size=0.2, random_state=42, stratify=y_s2)
m = LogisticRegression(max_iter=500, class_weight='balanced')
m.fit(X_tr, y_tr)
f1_leak = f1_score(y_te, m.predict(X_te))
print(f"F1 avec target encoding AVEC fuite : {f1_leak:.4f}")

In [ ]:
# TODO : Corrigez la fuite — calculez l'encodage UNIQUEMENT sur les données d'entraînement

# 1. Séparez d'abord (test_size=0.2, random_state=42, stratify=y_s2)
# df_train, df_test = ... (splittez df_s2 directement avec iloc ou via indices)

# Astuce : splittez les indices
# idx = np.arange(len(df_s2))
# idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_s2)
# df_train = df_s2.iloc[idx_train]
# df_test  = df_s2.iloc[idx_test]

# 2. Calculez l'encodage SEULEMENT sur df_train
# encoding_train = df_train.groupby('age')['readmit_30'].mean()

# 3. Appliquez cet encodage sur train ET test (sans recalculer)
# df_train['age_enc'] = df_train['age'].map(encoding_train)
# df_test['age_enc']  = df_test['age'].map(encoding_train)

# 4. Gérez les modalités inconnues dans le test (NaN si groupe absent du train)
# df_test['age_enc'].fillna(encoding_train.mean(), inplace=True)

# 5. Entraînez et évaluez
# feats_ok = FEATS_NUM + ['age_enc']
# ...
# print(f"F1 avec target encoding SANS fuite : {f1_ok:.4f}")

---
## Scénario 3 : Fuite Patient (Group Leakage)

Ce dataset contient **101 766 séjours** hospitaliers pour seulement **71 518 patients uniques**.  
Un même patient peut donc apparaître plusieurs fois (hospitalisations répétées).

**Le problème :** si on fait un split aléatoire, le même patient peut se retrouver à la fois dans le train et dans le test.  
Le modèle apprend alors des **patterns spécifiques à un patient** (âge, pathologies chroniques) plutôt que des patterns généralisables.

### Analogy
> Préparer un examen en révisant les **copies corrigées d'élèves qui repassent le même examen**. Les notes semblent bonnes, mais le modèle a juste mémorisé les patients.

In [ ]:
# Analysons les patients multi-séjours
sejours_par_patient = df_num.groupby('patient_nbr').size()

print("Distribution du nombre de séjours par patient :")
print(sejours_par_patient.value_counts().sort_index().head(10))
print(f"\nPatients avec ≥ 2 séjours : {(sejours_par_patient >= 2).sum()} "
      f"({(sejours_par_patient >= 2).mean():.1%} des patients)")

# Avec un split aléatoire 80/20, combien de patients sont dans les deux sets ?
idx_all = df_num.index.values
idx_train_rand, idx_test_rand = train_test_split(idx_all, test_size=0.2, random_state=42)

patients_train = set(df_num.loc[idx_train_rand, 'patient_nbr'])
patients_test  = set(df_num.loc[idx_test_rand,  'patient_nbr'])
overlap = patients_train & patients_test

print(f"\nAvec split ALÉATOIRE :")
print(f"  Patients dans le train : {len(patients_train)}")
print(f"  Patients dans le test  : {len(patients_test)}")
print(f"  Patients dans LES DEUX : {len(overlap)} ← FUITE !")

In [ ]:
# ❌ Split aléatoire (avec fuite patient)
X_s3 = df_num[FEATS_NUM].values
y_s3 = df_num['readmit_30'].values
groups = df_num['patient_nbr'].values

X_train_rand = X_s3[np.isin(df_num.index, idx_train_rand)]
X_test_rand  = X_s3[np.isin(df_num.index, idx_test_rand)]
y_train_rand = y_s3[np.isin(df_num.index, idx_train_rand)]
y_test_rand  = y_s3[np.isin(df_num.index, idx_test_rand)]

model_rand = LogisticRegression(max_iter=500, class_weight='balanced')
model_rand.fit(X_train_rand, y_train_rand)
f1_rand = f1_score(y_test_rand, model_rand.predict(X_test_rand))
print(f"F1 (split aléatoire, AVEC fuite patient) : {f1_rand:.4f}")

In [ ]:
# TODO : Corrigez avec un split par patient (GroupShuffleSplit)
# GroupShuffleSplit garantit qu'un patient est soit dans le train, soit dans le test, jamais les deux.

# gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
# train_idx, test_idx = next(gss.split(X_s3, y_s3, groups=groups))

# X_train_grp, X_test_grp = X_s3[train_idx], X_s3[test_idx]
# y_train_grp, y_test_grp = y_s3[train_idx], y_s3[test_idx]

# model_grp = LogisticRegression(max_iter=500, class_weight='balanced')
# model_grp.fit(X_train_grp, y_train_grp)
# f1_grp = f1_score(y_test_grp, model_grp.predict(X_test_grp))
# print(f"F1 (split par patient, SANS fuite) : {f1_grp:.4f}")

# Vérification : y a-t-il encore des patients dans les deux sets ?
# patients_train_grp = set(groups[train_idx])
# patients_test_grp  = set(groups[test_idx])
# overlap_grp = patients_train_grp & patients_test_grp
# print(f"Patients dans les deux sets après GroupSplit : {len(overlap_grp)} ← Doit être 0 !")

---
## Synthèse des 3 Scénarios

Rassemblez les F1-scores obtenus et comparez-les visuellement.

In [ ]:
# TODO : Complétez ce dictionnaire avec vos résultats et affichez un graphique comparatif

resultats = {
    'Scén.1\nAvec fuite\nprétraitement': f1_bad,
    'Scén.1\nSans fuite\n(split first)': None,   # ← À compléter
    'Scén.1\nPipeline': None,                     # ← À compléter
    'Scén.2\nTarget enc.\nfuite': f1_leak,
    'Scén.2\nTarget enc.\ncorrect': None,          # ← À compléter
    'Scén.3\nSplit rand.\nfuite patient': f1_rand,
    'Scén.3\nGroupSplit\ncorrect': None,           # ← À compléter
}

# Filtrez les None avant d'afficher
# res_complet = {k: v for k, v in resultats.items() if v is not None}
# plt.figure(figsize=(12, 5))
# colors = ['#e74c3c' if 'fuite' in k else '#2ecc71' for k in res_complet]
# plt.bar(res_complet.keys(), res_complet.values(), color=colors, alpha=0.8, edgecolor='black')
# plt.ylabel('F1-score (classe réadmis <30j)')
# plt.title('Comparaison des F1-scores : avec vs sans fuite de données')
# plt.tight_layout()
# plt.show()

---
## Checklist Anti-Fuite

| Question | Risque |
|---|---|
| Mon scaler/encodeur est-il fitté **avant** le split ? | Fuite de prétraitement |
| Mes statistiques cibles (target encoding) sont-elles calculées sur le dataset complet ? | Fuite d'encodage |
| Des individus apparaissent-ils à la fois dans train et test ? | Fuite patient/groupe |
| Mon F1 semble trop beau pour être vrai ? | Suspicion de fuite |

### Questions de réflexion
1. Pourquoi utiliser un **Pipeline** protège-t-il contre la fuite de prétraitement y compris en cross-validation ?
2. Dans ce dataset hospitalier, quels autres types de fuite pourraient exister ?
   (Indice : cherchez les colonnes liées à ce qui se passe **pendant** et **après** l'hospitalisation)
3. Comment le `GroupShuffleSplit` se compare-t-il à un simple `train_test_split` en termes de variance de l'estimation ?